# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [1]:
!pip -q install transformers sentencepiece torch



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


C:\Users\Majed AL-Hajri\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [3]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 4373.08it/s]

Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights:   4%|▎         | 7/190 [00:00<00:03, 53.96it/s]

Loading weights:   7%|▋         | 13/190 [00:00<00:13, 13.48it/s]

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 218.56it/s]


[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


C:\Users\Majed AL-Hajri\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\generation\utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [5]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   5%|▍         | 5/104 [00:00<00:03, 27.43it/s]

Loading weights:   8%|▊         | 8/104 [00:00<00:03, 25.11it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 300.25it/s]

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

**Answer:**

| # | Task | Most suitable family | Example model | Why |
|---|---|---|---|---|
| 1 | Sentiment classification | **Encoder-only** | BERT, RoBERTa | Needs to *understand* the whole sentence and output a label — no text generation required. |
| 2 | Machine translation | **Encoder–decoder** | T5, BART, MarianMT | Encoder reads the source sentence, decoder generates the target sentence (input → different output text). |
| 3 | Open-ended text generation | **Decoder-only** | GPT family | Generates text token-by-token, each token conditioned on the previous ones. |
| 4 | Named Entity Recognition | **Encoder-only** | BERT | Token classification: every token gets a label using context from both sides. |
| 5 | Text summarization | **Encoder–decoder** | T5, BART | Reads a long input and generates a shorter, new output text. |

## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [6]:
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers.",
]

results = classifier(sentences)

for sentence, result in zip(sentences, results):
    print(f"Sentence: {sentence}")
    print(f"   Label: {result['label']}, Score: {result['score']:.4f}\n")

Sentence: The movie was excellent.
   Label: POSITIVE, Score: 0.9999

Sentence: The lecture was difficult to understand.
   Label: NEGATIVE, Score: 0.9997

Sentence: I enjoyed learning about transformers.
   Label: POSITIVE, Score: 0.9997



## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [7]:
paragraph = ("Natural language processing is a field of artificial intelligence that helps computers "
             "understand, interpret and generate human language. It is used in many everyday applications "
             "such as machine translation, chatbots, spam filters and voice assistants. Modern NLP systems "
             "are based on transformer models that are trained on very large amounts of text.")

prompt = "Summarize: " + paragraph

inputs = tokenizer(prompt, return_tensors="pt")                   # encoder input
output = model.generate(**inputs, max_new_tokens=60, num_beams=4)  # decoder generates the summary

print("Original paragraph:\n", paragraph)
print("\nSummary:\n", tokenizer.decode(output[0], skip_special_tokens=True))

Original paragraph:
 Natural language processing is a field of artificial intelligence that helps computers understand, interpret and generate human language. It is used in many everyday applications such as machine translation, chatbots, spam filters and voice assistants. Modern NLP systems are based on transformer models that are trained on very large amounts of text.

Summary:
 Learn how to use natural language processing. Learn how to use natural language processing. Learn how to use natural language processing. Learn how to use natural language processing. Learn how to use natural language processing. Learn how to use natural language processing. Learn how to use natural language processing. Learn how to use



## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


**Answer:**

A chatbot's job is to **generate a reply one token at a time**, and that is exactly what a decoder-only model is trained to do: predict the next token from all the previous tokens (causal / autoregressive language modeling). The whole conversation — the system instructions, the user's messages and the model's earlier replies — can simply be placed in the context as one sequence, and the model continues it with the next answer. Because it is trained on huge amounts of text with this single simple objective, it scales very well and learns grammar, facts and reasoning patterns that make its answers fluent. It does not need a separate encoder because the input and the output live in the same sequence, which makes it a natural fit for open-ended, multi-turn conversation.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


In [8]:
generator = pipeline("text-generation", model="distilgpt2")
prompt = "Natural language processing is"

# With a fixed seed -> the same "random" output every time
for run in range(2):
    set_seed(42)
    out = generator(prompt, max_new_tokens=25, do_sample=True, temperature=0.9)
    print(f"[seed 42, run {run + 1}]", out[0]["generated_text"].replace("\n", " "))

print()

# Without set_seed -> a different output on each run
for run in range(3):
    out = generator(prompt, max_new_tokens=25, do_sample=True, temperature=0.9)
    print(f"[no seed, run {run + 1}]", out[0]["generated_text"].replace("\n", " "))

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 3596.61it/s]

[seed 42, run 1] Natural language processing is an area that has gained significant popularity since the 1970s. Language processing in this area is currently under way. Several studies have


[seed 42, run 2] Natural language processing is an area that has gained significant popularity since the 1970s. Language processing in this area is currently under way. Several studies have



[no seed, run 1] Natural language processing is common in most countries.          It looks like the government is using the English version


[no seed, run 2] Natural language processing is a process that is highly involved in the translation of the languages and their meanings to those of each other. This has been a


[no seed, run 3] Natural language processing is a very complex topic, but it is a big part of the application and is an extremely practical and easy to understand approach.


**Answer — why do we get different outputs for the same prompt?**

With `do_sample=True` the model does **not** always pick the most likely next token (that would be greedy decoding, `do_sample=False`, which always gives the same output). Instead, it computes a probability distribution over the vocabulary and **randomly samples** the next token from it. Each run therefore makes different random choices, and since every new token depends on the previous ones, a small difference early on leads to a completely different continuation. `set_seed(42)` fixes the random number generator, so the "random" choices are reproducible; removing it gives different outputs every time. Parameters such as `temperature`, `top_k` and `top_p` control how much randomness is allowed.